# 03 — Export

Aggregates lot-level `far_gap` up to block-group level, and exports two GeoJSON files consumed by `web/map.js`:
- `bushwick_far_gap.geojson` — block group polygons, for the choropleth fill
- `bushwick_far_lots.geojson` — individual lots, for the clickable point layer (address, FAR, Street View, ACRIS)

Both are reprojected back to EPSG:4326 (plain WGS84 lat/lon) before export — GeoJSON's spec requires WGS84, but the working CRS since notebook 01 has been the block groups' NAD83.


In [1]:
import geopandas as gpd


In [2]:
gdf_joined = gpd.read_file('cache/bw_joined.geojson')
gdf_bg = gpd.read_file('cache/bg_kings.geojson')

In [3]:
df_bg_far = (
    gdf_joined.groupby('GEOID')
    .agg(far_gap=('far_gap', 'mean'), lot_count=('far_gap', 'size'))
    .reset_index()
)
print(df_bg_far.sort_values('far_gap', ascending=False).head())

           GEOID   far_gap  lot_count
3   360470391001  1.805610         83
67  360470433001  1.479535         43
25  360470407001  1.170000          4
49  360470423001  1.149440        232
40  360470417003  1.129157         83


In [4]:
# Block group polygons for the choropleth. Inner join — only keep block
# groups that actually matched a Bushwick lot (same reasoning as filtering
# out the NaN rows in Day 18 before plotting: keeps the map zoomed to the
# area with real data instead of all of Kings County).
gdf_bg_far = gdf_bg.merge(df_bg_far, on='GEOID', how='inner')
gdf_bg_far = gdf_bg_far[['GEOID', 'far_gap', 'lot_count', 'geometry']].to_crs('EPSG:4326')

gdf_bg_far.to_file('../web/assets/bushwick_far_gap.geojson', driver='GeoJSON')
print(gdf_bg_far.shape)

(92, 4)


In [5]:
# Individual lots for the clickable point layer.
lot_cols = ['address', 'builtfar', 'residfar', 'far_gap', 'BBL', 'borough',
            'Tax block', 'Tax lot', 'GEOID', 'geometry']
gdf_lots = gdf_joined[lot_cols].copy()
gdf_lots = gdf_lots.rename(columns={'Tax block': 'block', 'Tax lot': 'lot'})
gdf_lots = gdf_lots.to_crs('EPSG:4326')

gdf_lots.to_file('../web/assets/bushwick_far_lots.geojson', driver='GeoJSON')
print(gdf_lots.shape)

(11154, 10)


# 04 — Print sheet for the Hancock/Jefferson walk

A printable CSV for one physical block: Hancock Street and Jefferson Avenue,
between Wyckoff Avenue and Irving Avenue. In PLUTO this is a single tax block
(3393) — Hancock-fronting lots on one side, Jefferson-fronting lots on the
other. Corner lots addressed directly on Wyckoff/Irving are excluded since
the walk is Hancock and Jefferson specifically.

Output: `outputs/hancock_jefferson_far_walk.csv`

In [ ]:
# Tax block 3393 = the block bounded by Hancock St, Jefferson Ave, Wyckoff Ave, Irving Ave.
# Keep only Hancock/Jefferson-fronting lots with a real street number (drops the two Z7
# easement slivers on this block, which have no address and no building).
on_hancock_or_jefferson = gdf_joined['address'].str.upper().str.contains('HANCOCK STREET|JEFFERSON AVENUE')
has_house_number = gdf_joined['address'].str.match(r'^\d')
walk_block = gdf_joined[(gdf_joined['Tax block'] == 3393) & on_hancock_or_jefferson & has_house_number].copy()

walk_cols = {
    'address': 'Address',
    'Tax block': 'Block',
    'Tax lot': 'Lot',
    'BBL': 'BBL',
    'builtfar': 'Built FAR',
    'residfar': 'Max FAR',
    'far_gap': 'Unbuilt FAR',
}
df_walk = walk_block[list(walk_cols)].rename(columns=walk_cols)
df_walk[['Built FAR', 'Max FAR', 'Unbuilt FAR']] = df_walk[['Built FAR', 'Max FAR', 'Unbuilt FAR']].round(2)
df_walk = df_walk.sort_values('Address')

df_walk.to_csv('../../../outputs/hancock_jefferson_far_walk.csv', index=False)
print(df_walk.shape)
df_walk